# 03 Baseline Modeling
This notebook trains and evaluates the baseline models requested.

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import time

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, log_loss, f1_score, classification_report


In [2]:

# Load data
print("Loading data...")
X_train = pd.read_csv('../data/processed/train_X.csv')
y_train = pd.read_csv('../data/processed/train_y.csv').values.ravel()

X_val = pd.read_csv('../data/processed/val_X.csv')
y_val = pd.read_csv('../data/processed/val_y.csv').values.ravel()

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")


Loading data...
X_train shape: (9600, 31), y_train shape: (9600,)
X_val shape: (2400, 31), y_val shape: (2400,)


In [3]:

# Define models
models = {
    'KNN': KNeighborsClassifier(),
    'Logistic_Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision_Tree': DecisionTreeClassifier(random_state=42),
    'Naive_Bayes': GaussianNB(),
    'Random_Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBOOST': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1)
}

# Directories for saving
os.makedirs('../results/models', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)


In [4]:

results = []

for name, model in models.items():
    print(f"\n[{name}] Training...")
    start_time = time.time()
    
    # Train
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predict on validation set
    y_pred = model.predict(X_val)
    y_pred_proba = model.predict_proba(X_val)
    
    # Calculate metrics
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    loss = log_loss(y_val, y_pred_proba)
    
    # Save model
    model_path = f'../results/models/{name}.pkl'
    joblib.dump(model, model_path)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Macro_F1': f1,
        'Log_Loss': loss,
        'Train_Time(s)': round(train_time, 2),
        'Model_File': model_path
    })
    
    print(f"[{name}] Acc: {acc:.4f} | F1: {f1:.4f} | LogLoss: {loss:.4f} | Saved to: {model_path}")



[KNN] Training...


[KNN] Acc: 0.8196 | F1: 0.5511 | LogLoss: 2.4531 | Saved to: ../results/models/KNN.pkl

[Logistic_Regression] Training...


[Logistic_Regression] Acc: 0.8317 | F1: 0.5481 | LogLoss: 0.4458 | Saved to: ../results/models/Logistic_Regression.pkl

[Decision_Tree] Training...
[Decision_Tree] Acc: 0.7833 | F1: 0.5544 | LogLoss: 7.8095 | Saved to: ../results/models/Decision_Tree.pkl

[Naive_Bayes] Training...
[Naive_Bayes] Acc: 0.5854 | F1: 0.4622 | LogLoss: 2.2685 | Saved to: ../results/models/Naive_Bayes.pkl

[Random_Forest] Training...


[Random_Forest] Acc: 0.8479 | F1: 0.5997 | LogLoss: 0.5419 | Saved to: ../results/models/Random_Forest.pkl

[XGBOOST] Training...


C:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:13:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[XGBOOST] Acc: 0.8500 | F1: 0.6296 | LogLoss: 0.4110 | Saved to: ../results/models/XGBOOST.pkl


In [5]:

df_results = pd.DataFrame(results).sort_values('Log_Loss', ascending=True).reset_index(drop=True)
df_results.to_csv('../results/metrics/baseline_metrics.csv', index=False)

print("\n=== BASELINE MODELS SUMMARY ===")
display(df_results)



=== BASELINE MODELS SUMMARY ===


,Model,Accuracy,Macro_F1,Log_Loss,Train_Time(s),Model_File
0,XGBOOST,0.850000,0.629594,0.410999,0.52,../results/models/XGBOOST.pkl
1,Logistic_Regression,0.831667,0.548107,0.445753,0.33,../results/models/Logistic_Regression.pkl
2,Random_Forest,0.847917,0.599671,0.541900,0.28,../results/models/Random_Forest.pkl
3,Naive_Bayes,0.585417,0.462245,2.268461,0.01,../results/models/Naive_Bayes.pkl
4,KNN,0.819583,0.551104,2.453111,0.00,../results/models/KNN.pkl
5,Decision_Tree,0.783333,0.554365,7.809458,0.17,../results/models/Decision_Tree.pkl
